# 03 — Federated K-Means Runs

Run federated k-means via FeatureCloud and aggregate results.

**What this notebook does:**
1. Prepare per-site input directories for the FeatureCloud fc_kmeans app.
2. Launch federated k-means tests (requires Docker + FeatureCloud CLI).
3. OR aggregate existing federated outputs (if FeatureCloud is unavailable).
4. Join federated cluster assignments with metadata.

**Prerequisites:**
- Run notebook 01 first (data preparation).
- For live runs: Docker running + FeatureCloud CLI installed.
- For aggregation only: existing zip results in `real_datasets/<dataset>/inputs/*/tests/`.

## Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    discover_clients,
    load_feature_matrix,
    load_metadata,
    prepare_variant_inputs,
    aggregate_fed_clusters,
)
from evaluation_utils.featurecloud_kmeans_utils import (
    ensure_app_image,
    run_single_federated_variant,
    aggregate_existing_federated_output,
)

## Configuration

In [2]:
DATASETS = ["proteomics", "microarray", "proteomics_multibatch", "ccRCC_proteomics"]

# Set to True to launch a real FeatureCloud test.
# Set to False to only aggregate existing zip results.
RERUN_FEDERATED = True

# FeatureCloud settings (only needed when RERUN_FEDERATED=True)
APP_IMAGE = "fc_kmeans_upd"
APP_SOURCE_DIR = NOTEBOOK_DIR.parent / "federated_kmeans_upd"
CONTROLLER_HOST = "http://localhost:8000"
QUERY_INTERVAL = 5
TIMEOUT = 1800

OUTPUT_ROOT = NOTEBOOK_DIR

## Prepare FeatureCloud Inputs

Build per-site input directories with `intensities.tsv`, `design.tsv`,
and `config_kmeans.yml` — the format required by the fc_kmeans app.

This step creates `real_datasets/<dataset>/inputs/{before,corrected}/<site>/`
from the prepared matrices saved by notebook 01. It always runs (fast and idempotent)
so that the federated step below has up-to-date inputs.

In [3]:
# Always generate per-site input directories (fast, idempotent).
# These are needed for both fresh federated runs and for aggregating existing results.
configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    prepared_dir = ds_dir / "prepared"

    # Load prepared data from notebook 01
    before = load_feature_matrix(prepared_dir / "before_matrix.tsv")
    corrected = load_feature_matrix(prepared_dir / "corrected_matrix.tsv")
    metadata = pd.read_csv(prepared_dir / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: preparing FC inputs for k={k_values}")
    print(f"{'='*60}")

    # Generate per-site input directories for the fc_kmeans app
    before_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="before",
        matrix=before, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    corrected_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="corrected",
        matrix=corrected, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    print(f"Created: {before_dir}")
    print(f"Created: {corrected_dir}")


proteomics: preparing FC inputs for k=[2, 5]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics/inputs/corrected

microarray: preparing FC inputs for k=[2, 6]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/microarray/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/microarray/inputs/corrected

proteomics_multibatch: preparing FC inputs for k=[4]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics_multibatch/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics_multibatch/inputs/corrected

ccRCC_proteomics: prepar

## Run or Aggregate Federated K-Means

When `RERUN_FEDERATED=True`, this launches a FeatureCloud test for each variant.
When `False`, it aggregates existing zip results from previous runs.

In [4]:
for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    metadata = pd.read_csv(ds_dir / "prepared" / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)
    client_names = [c.name for c in clients]

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: {'running' if RERUN_FEDERATED else 'aggregating'} federated k-means")
    print(f"{'='*60}")

    for variant in ["before", "corrected"]:
        variant_input_dir = ds_dir / "inputs" / variant

        try:
            output_path = run_single_federated_variant(
                dataset_name=ds_name,
                variant_label=variant,
                variant_input_dir=variant_input_dir,
                dataset_root=ds_dir,
                metadata=metadata,
                client_names=client_names,
                k_values=k_values,
                app_image=APP_IMAGE,
                controller_host=CONTROLLER_HOST,
                query_interval=QUERY_INTERVAL,
                timeout=TIMEOUT,
                keep_extracted=False,
                aggregate_only=not RERUN_FEDERATED,
            )
            print(f"  {variant}: saved to {output_path}")
        except FileNotFoundError as e:
            print(f"  {variant}: SKIPPED — {e}")
        except RuntimeError as e:
            print(f"  {variant}: FAILED — {e}")


proteomics: running federated k-means
[proteomics] Starting FeatureCloud controller for 'before' data
Killing leftover containers...
Starting the FeatureCloud controller with the data directory /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics/inputs/before


Downloading...: 206it [00:09, 22.13it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[proteomics] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_627832715', 'containerID': 'dd30bd1fe3175ce8', 'coordinator': False, 'frontendUrl': '/app-traffic/01ceb6a771/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_551182665', 'containerID': 'f1003093abae7391', 'coordinator': False, 'frontendUrl': '/app-traffic/c329c6cc9d/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_484431785', 'containerID': '3139f2a84eda9075', 'coordinator': False, 'frontendUrl': '/app-traffic/061ea68419/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_480710991', 'containerID': '0bb93ce815de0a17', 'coordinator': True, 'frontendUrl': '/app-traffic/c1ea8be8b9/', 

Downloading...: 3it [00:00, 1943.01it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[proteomics] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_519612977', 'containerID': '7a31edcab3a05e2e', 'coordinator': True, 'frontendUrl': '/app-traffic/b274a3ec03/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_556750935', 'containerID': '21f3e79b0ba5a56a', 'coordinator': False, 'frontendUrl': '/app-traffic/3ebd9c6436/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_602375338', 'containerID': 'd4ca556cfaef525f', 'coordinator': False, 'frontendUrl': '/app-traffic/79242f1ca9/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_957676913', 'containerID': '20f95a20e4d7290b', 'coordinator': False, 'frontendUrl': '/app-traffic/4016ae18e1/

...

Controller started successfully, waiting 5s to ensure the controller is properly running!
[proteomics_multibatch] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_416240321', 'containerID': '3a3c3abeac3cca42', 'coordinator': False, 'frontendUrl': '/app-traffic/44fb58be35/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_520924160', 'containerID': '7e28e42402109c92', 'coordinator': False, 'frontendUrl': '/app-traffic/a85a09d352/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_592500095', 'containerID': '46530a33f9c2e50c', 'coordinator': False, 'frontendUrl': '/app-traffic/c57fc99074/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_902368805', 'containerID': '4f53e265d300dc08', 'coordinator': True, 'frontendUrl': '/app-traffic/

Downloading...: 3it [00:00, 1506.03it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ccRCC_proteomics] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_244927952', 'containerID': 'd23dbce4ebcaf977', 'coordinator': False, 'frontendUrl': '/app-traffic/9f76c8e716/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_511078614', 'containerID': 'd05a875af36d9b61', 'coordinator': False, 'frontendUrl': '/app-traffic/4bdbba115c/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_649188440', 'containerID': 'b643607ae6c3f51e', 'coordinator': True, 'frontendUrl': '/app-traffic/f366d1d652/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[ccRCC_proteomics] Federated output saved: /home/yuliya-cosybio/re

Downloading...: 3it [00:00, 2027.54it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ccRCC_proteomics] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_273503049', 'containerID': '3f23f18016d96039', 'coordinator': False, 'frontendUrl': '/app-traffic/8eeb3d7a2d/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_307966117', 'containerID': '8e7c8992ed6a932b', 'coordinator': True, 'frontendUrl': '/app-traffic/899392d3bd/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_341631080', 'containerID': '2deb72222c4f1224', 'coordinator': False, 'frontendUrl': '/app-traffic/9aa9ce183a/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[ccRCC_proteomics] Federated output saved: /home/yuliya-cosybio

## Verify Outputs

Check that federated metadata files were produced.

In [5]:
for ds_name in DATASETS:
    run_dir = OUTPUT_ROOT / ds_name / "kmeans_res" / "runs"
    for fname in ["1_metadata_before_fedclusters.tsv", "1_metadata_after_fedclusters.tsv"]:
        p = run_dir / fname
        if p.exists():
            df = pd.read_csv(p, sep="\t")
            print(f"{ds_name}/{fname}: {df.shape[0]} samples, cols={list(df.columns)}")
        else:
            print(f"{ds_name}/{fname}: NOT FOUND")

proteomics/1_metadata_before_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
proteomics/1_metadata_after_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
microarray/1_metadata_before_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
microarray/1_metadata_after_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
proteomics_multibatch/1_metadata_before_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
proteomics_multibatch/1_metadata_after_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
ccRCC_proteomics/1_metadata_before_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3clusters']
ccRCC_proteomics/1_metadata_after_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3c